In [ ]:
import pandas as pd
import os
import os.path as osp
import numpy as np
import json, requests, argparse
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

import config_logger

In [ ]:
try:
    PROJECT_ROOT = osp.abspath(osp.join(osp.dirname(__file__),'..'))
except:
    PROJECT_ROOT = osp.abspath(osp.join(os.getcwd(),'..'))

logger = config_logger.get_logger(__name__)
logger.info(PROJECT_ROOT)

2026-04-17 16:25:41 [INFO] __main__ | /home/bfeldmann/projects/formation_mlops/projet_meteoAUS


In [ ]:
_description = f""" 
Télécharge les données météo Australien depuis le site bom.gov.au

"""
parser = argparse.ArgumentParser(description='\n'.join([_description]))
parser.add_argument('--city', type=str, required=True, help="City name to get weather report")
parser.add_argument('--maxlag', type=str, required=False, default=3, help="Maximum lag computed in features model")

In [3]:
def get_stations_infos():
    stations_file = osp.join(PROJECT_ROOT,'src','stations_infos.json')
    if not osp.exists(stations_file):
        logger.error(f'{stations_file} not found !')
        raise FileNotFoundError
    
    logger.info(f'Load {osp.basename(stations_file)}')
    with open(stations_file) as src:
        stations_data = json.load(src)
    return stations_data

In [10]:
class DailyWeatherDATA:
    def __init__(self):
        self.url_daily_weather = 'https://www.bom.gov.au/climate/dwo/{year_month}/text/{bom_id}.{year_month}.csv'

        self.stations_infos = get_stations_infos()

    def get_available_cities(self):
        return list(self.stations_infos.keys())
    
    def get_station_info(self, city):
        station_info = self.stations_infos.get(city,None)

        if station_info is None:
            logger.error(f'Unknown city: {city}')
            raise ValueError("Unknown city")
        
        return station_info
    
    def get_station_date(self, city):
        station_info = self.get_station_info(city)        
        now = datetime.now(ZoneInfo(station_info['timezone']))
        return now
            
    def get_url(self,city, time='last'):
        station_info = self.get_station_info(city)
        
        if time=='last':
            now = self.get_station_date(city)
            date_month = now.strftime("%Y%m")
        elif len(time)==6 and time[0:2]=='20':
            date_month = time
        else:
            logger.error(f'Not valid time, expected YYYYMM format but got {time}')
            raise ValueError(f'{time}')
        
        return self.url_daily_weather.format(year_month=date_month, bom_id=station_info['bom_id'])
    
    def get_report(self,csv_url, out_dir):
        headers = {
                "User-Agent": "Mozilla/5.0"
            }
        
        temp_filename_split = osp.basename(csv_url).split('.')
        out_path = osp.join(out_dir, f'{temp_filename_split[0]}_{temp_filename_split[1]}.{temp_filename_split[2]}')

        if not osp.exists(out_path):
            logger.info('Download weather report')
            response = requests.get(csv_url, headers=headers)
            response.raise_for_status()

            with open(out_path, "wb") as f:
                f.write(response.content)

        return out_path

    def cleaning_report(self,in_file, out_dir):
        raw_cols = ['Date', 'Minimum temperature (°C)', 'Maximum temperature (°C)',
                    'Rainfall (mm)', 'Evaporation (mm)', 'Sunshine (hours)',
                    'Direction of maximum wind gust ', 'Speed of maximum wind gust (km/h)',
                    'Time of maximum wind gust', '9am Temperature (°C)',
                    '9am relative humidity (%)', '9am cloud amount (oktas)',
                    '9am wind direction', '9am wind speed (km/h)', '9am MSL pressure (hPa)',
                    '3pm Temperature (°C)', '3pm relative humidity (%)',
                    '3pm cloud amount (oktas)', '3pm wind direction',
                    '3pm wind speed (km/h)', '3pm MSL pressure (hPa)']

        dict_cols = {
            'Date':'datetime', 'MinTemp': 'float32', 'MaxTemp': 'float32', 'Rainfall': 'float32', 'Evaporation': 'float32',
            'Sunshine': 'float32', 'WindGustDir': 'string', 'WindGustSpeed': 'float32', 'TimeMaxWindGust': 'string',
            'Temp9am': 'float32', 'Humidity9am': 'int16', 'Cloud9am': 'float32', 'WindDir9am': 'string',
            'WindSpeed9am': 'float32', 'Pressure9am': 'float32', 'Temp3pm': 'float32', 'Humidity3pm': 'int16',
            'Cloud3pm': 'float32', 'WindDir3pm': 'string', 'WindSpeed3pm': 'float32', 'Pressure3pm': 'float32'
            }
        dict_raintoday = {0: 'No', 1: 'Yes'}
        
        logger.info('Cleaning weather report')
        df = pd.read_csv(in_file, delimiter=',', skiprows=7, encoding='latin-1')
        df = df.drop(columns=df.columns[0])
        
        if all(df.columns == raw_cols):
            df.columns = dict_cols.keys()
            
        for col,dtype in dict_cols.items():
            if dtype in ('int16', 'float32'):
                df[col] = pd.to_numeric(df[col], errors='coerce').astype(dtype)
                
            elif dtype=='datetime':
                df[col] = pd.to_datetime(df[col], errors='coerce')
                
            else:
                df[col] = df[col].astype(dtype)
        
        df = df.replace(
            [pd.NA, None, "NaN", "nan", "NAN", "NA", "N/A", "", " ", "null", "None"],
            np.nan
        )
        df = df.drop(columns='TimeMaxWindGust')
        df['RainToday'] = df['Rainfall'].apply(lambda x: dict_raintoday.get(x>1, None))

        filename = osp.basename(in_file)
        out_file = osp.join(out_dir, f'clean_{filename}')     
        df.to_csv(out_file, sep=',', header=True, index=False)
        return out_file

In [ ]:
def run(city, maxlag=3):
    raw_report_dir = osp.join(PROJECT_ROOT, 'raw_data')
    clean_report_dir = osp.join(PROJECT_ROOT, 'data')
    
    process = DailyWeatherDATA()

    station_date = process.get_station_date(city)
    station_lag_date = station_date - timedelta(days=maxlag)

    if station_date.strftime("%m") != station_lag_date.strftime("%m"):
        month_list = [station_date.strftime("%Y%m"), station_lag_date.strftime("%Y%m")]
    else:
        month_list = [station_date.strftime("%Y%m")]
    
    list_clean_report = []
    for yearmonth in month_list:
        logger.info(f'Process {yearmonth}')
        url = process.get_url(city, yearmonth)
        raw_file = process.get_report(url, raw_report_dir)
        list_clean_report.append(process.cleaning_report(raw_file, clean_report_dir))
    
    if len(list_clean_report) > 1:
        logger.info('Merge all reports')
        list_df = [pd.read_csv(i) for i in list_clean_report]
        merge_df = pd.concat(list_df)

        merge_df['Date'] = pd.to_datetime(merge_df['Date'], errors='coerce')
        merge_df = merge_df.sort_values(by='Date')
        merge_df.to_csv(list_clean_report[0], sep=',', header=True, index=False, mode='w')
    
    logger.info('Finished !')

In [ ]:
if __name__ == "__main__":
    # Parse command line arguments
    kwargs = parser.parse_args()
    print("\n")
    print("get Daily Weather Report command line arguments: \n")
    print(json.dumps(vars(kwargs), indent=1)) # Pretty print dictionary
    print("\n")

    # Run
    run(**vars(kwargs))

# TEST

In [7]:
data_file = osp.join(PROJECT_ROOT,'data','weatherAUS.csv')
df = pd.read_csv(data_file)
np.unique(df['Location'])

array(['Adelaide', 'Albany', 'Albury', 'AliceSprings', 'BadgerysCreek',
       'Ballarat', 'Bendigo', 'Brisbane', 'Cairns', 'Canberra', 'Cobar',
       'CoffsHarbour', 'Dartmoor', 'Darwin', 'GoldCoast', 'Hobart',
       'Katherine', 'Launceston', 'Melbourne', 'MelbourneAirport',
       'Mildura', 'Moree', 'MountGambier', 'MountGinini', 'Newcastle',
       'Nhil', 'NorahHead', 'NorfolkIsland', 'Nuriootpa', 'PearceRAAF',
       'Penrith', 'Perth', 'PerthAirport', 'Portland', 'Richmond', 'Sale',
       'SalmonGums', 'Sydney', 'SydneyAirport', 'Townsville',
       'Tuggeranong', 'Uluru', 'WaggaWagga', 'Walpole', 'Watsonia',
       'Williamtown', 'Witchcliffe', 'Wollongong', 'Woomera'],
      dtype=object)

In [ ]:
raw_report_dir = osp.join(PROJECT_ROOT, 'raw_data')
clean_report_dir = osp.join(PROJECT_ROOT, 'data')
city = 'Canberra'
max_lag = 3

process = DailyWeatherDATA()

station_date = process.get_station_date(city)
station_lag_date = station_date - timedelta(days=max_lag)

if station_date.strftime("%m") != station_lag_date.strftime("%m"):
    month_list = [station_date.strftime("%Y%m"), station_lag_date.strftime("%Y%m")]
else:
    month_list = [station_date.strftime("%Y%m")]

list_clean_report = []
for yearmonth in month_list:
    logger.info(f'Process {yearmonth}')
    url = process.get_url(city, yearmonth)
    raw_file = process.get_report(url, raw_report_dir)
    list_clean_report.append(process.cleaning_report(raw_file, clean_report_dir))



2026-04-17 16:42:05 [INFO] __main__ | Load stations_infos.json
2026-04-17 16:42:05 [INFO] __main__ | Process 202604
2026-04-17 16:42:05 [INFO] __main__ | Cleaning weather report


In [18]:
df = pd.read_csv(list_clean_report[0])
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

df = df.sort_values(by='Date')
df.head()
# df = df.sort_values(by='Date')
# df.head()

,Date,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,Temp9am,Humidity9am,...,WindDir9am,WindSpeed9am,Pressure9am,Temp3pm,Humidity3pm,Cloud3pm,WindDir3pm,WindSpeed3pm,Pressure3pm,RainToday
0,2026-04-01,6.3,26.7,0.0,NaN,NaN,NaN,NaN,12.8,87,...,SE,9.0,1023.9,25.9,32,NaN,NW,17.0,1022.1,No
1,2026-04-02,9.5,NaN,0.0,NaN,NaN,NaN,NaN,14.5,85,...,N,6.0,1025.1,27.4,32,NaN,NW,19.0,1021.8,No
2,2026-04-03,7.6,25.0,0.0,NaN,NaN,E,41.0,13.6,88,...,NaN,NaN,1023.9,24.4,38,NaN,NE,13.0,1022.0,No
3,2026-04-04,11.6,17.3,0.0,NaN,NaN,ENE,28.0,13.2,81,...,NE,9.0,1027.2,15.8,67,8.0,NE,11.0,1025.9,No
4,2026-04-05,5.2,20.0,0.0,NaN,NaN,ENE,30.0,13.8,75,...,SSE,6.0,1027.8,17.6,57,3.0,NNE,11.0,1023.8,No
